# Anti-Money Laundering (AML) Baseline: Traditional ML on the Elliptic Bitcoin Dataset

This notebook establishes a baseline for traditional Machine Learning models (Random Forest and XGBoost) applied to the **Elliptic Bitcoin Dataset**. 

In the context of the Bachelor's Thesis comparing Traditional ML vs. Graph Neural Networks (GNNs) for fraud detection, this experiment demonstrates the limitations of purely tabular approaches. 

While tree-based models excel at finding patterns in isolated node features, they are entirely "blind" to the network topology. In financial forensics and Anti-Money Laundering (AML), illicit actors often camouflage their behavior to mimic licit users at the local level. By stripping away the `edge_index` (the transaction flow between entities), I force the models to classify nodes purely based on local attributes, leading to a performance ceiling that my custom MEGA-PNA GNN architecture aims to shatter.

## 1. Imports

Importing PyTorch Geometric to fetch the dataset, and scikit-learn/XGBoost for the traditional ML pipeline. I extract the data into standard NumPy arrays, mimicking the pipeline used in my repository's `fraud-detect traditional` CLI.

In [ ]:
import warnings

import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch_geometric.datasets import EllipticBitcoinDataset
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

## 2. Data Loading and Tabular Extraction

The Elliptic dataset consists of a single massive graph. 
- `data.x`: Contains the 166 features for each transaction (local and aggregated features).
- `data.edge_index`: Contains the transaction links. **I will intentionally ignore this for the traditional ML baseline.**
- `data.y`: The labels. In PyG, 0 is Illicit, 1 is Licit, and 2 is Unknown. 

To standardise the evaluation for fraud detection, I will remap the classes so that **Illicit (Fraud) becomes the positive class (1)** and Licit becomes the negative class (0). I will also use the predefined temporal split (first 34 timesteps for training, the rest for testing) using the provided masks.

In [ ]:
# Load the dataset
print("Downloading and loading the Elliptic dataset...")
dataset = EllipticBitcoinDataset(root="./data/Elliptic")
data = dataset[0]

print(f"Total nodes: {data.num_nodes}")
print(f"Total edges: {data.num_edges} (IGNORED FOR TABULAR ML)")

# Extract node features
X = data.x.numpy()

# Remap labels: Illicit (0 in PyG) -> 1 (Fraud), Licit (1 in PyG) -> 0 (Normal)
# Unknown nodes (2) will be filtered out by the train/test masks
y_mapped = (data.y == 0).to(torch.long).numpy()

# Extract masks
train_mask = data.train_mask.numpy()
test_mask = data.test_mask.numpy()

# Create Tabular splits
X_train, y_train = X[train_mask], y_mapped[train_mask]
X_test, y_test = X[test_mask], y_mapped[test_mask]

print(f"\nTraining set size: {X_train.shape[0]} nodes")
print(f"Testing set size: {X_test.shape[0]} nodes")
print(f"Fraud ratio in train: {y_train.sum() / len(y_train):.4f}")

## 3. Training Traditional ML Models

I will train a **Random Forest** and an **XGBoost** classifier. These models mirror the ones supported in my repository's source code. Because they evaluate the feature matrix $X$ independently for each row, they represent my topology-agnostic baselines.

In [ ]:
# 1. Random Forest Classifier
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced"
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# 2. XGBoost Classifier
print("Training XGBoost...")
# scale_pos_weight helps with the heavy class imbalance
fraud_ratio = (len(y_train) - y_train.sum()) / y_train.sum()
xgb_model = XGBClassifier(
    n_estimators=100, random_state=42, eval_metric="logloss", scale_pos_weight=fraud_ratio
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

print("\nModels trained successfully.")

## 4. Evaluation 

The primary metric of interest in highly imbalanced fraud datasets is the **F1-score of the minority class (Illicit/Fraud)**. As expected, without the relational context of the graph, the F1-score struggles to cross the 0.65 - 0.70 threshold due to poor recall or excessive false positives caused by structural camouflage.

In [ ]:
print("=" * 50)
print("RANDOM FOREST RESULTS (Topology-Blind)")
print("=" * 50)
print(classification_report(y_test, rf_preds, target_names=["Licit", "Illicit (Fraud)"]))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_preds))

print("\n" + "=" * 50)
print("XGBOOST RESULTS (Topology-Blind)")
print("=" * 50)
print(classification_report(y_test, xgb_preds, target_names=["Licit", "Illicit (Fraud)"]))
print("Confusion Matrix:\n", confusion_matrix(y_test, xgb_preds))

# Highlight the specific metric for the Thesis comparison
rf_f1 = f1_score(y_test, rf_preds)
xgb_f1 = f1_score(y_test, xgb_preds)
print("\n" + "-" * 50)
print("Target Baseline to beat with GNNs:")
print(f"Max F1-Score (Illicit): {max(rf_f1, xgb_f1):.4f}")
print("-" * 50)

## 5. Conclusion

Both XGBoost and Random Forest cap out at a suboptimal F1-score for the illicit class. While they successfully identify obvious bad actors based on tabular heuristics, they fail to catch sophisticated money launderers who obscure their local features but remain structurally tied to criminal clusters in the network.